# Top 10 Architectures for Building AI Agents

A hands-on notebook that implements each major agent architecture pattern
from scratch using **OpenAI GPT-4o**. Every pattern includes a working
example so we can see the difference in behavior, tradeoffs, and when
to use each one.

---

| # | Architecture | Core Idea | Production Readiness |
|---|-------------|-----------|---------------------|
| 1 | **ReAct** | Think, Act, Observe, Repeat | High -- default choice |
| 2 | **Plan-Execute** | Plan first, then execute steps | High -- structured workflows |
| 3 | **Reflection / Self-Critique** | Do, Critique, Improve | Medium -- quality-critical tasks |
| 4 | **Multi-Agent Systems** | Specialized agents collaborating | Medium -- complex domains |
| 5 | **Tool-Augmented (Function Calling)** | LLM + structured tool APIs | High -- production standard |
| 6 | **Memory-Augmented** | Short-term + long-term + episodic | High -- stateful agents |
| 7 | **Event-Driven** | Triggered by external events | High -- real-world automation |
| 8 | **Graph-Based Execution** | Nodes + edges = deterministic flow | High -- LangGraph style |
| 9 | **Autonomous Loop** | Run until goal is achieved | Low -- needs heavy guardrails |
| 10 | **Hierarchical** | Manager delegates to sub-agents | Medium -- enterprise scale |

In [ ]:
# -- Setup --
!pip install -q openai

import os, json, time, textwrap
from typing import List, Dict, Any, Callable
from collections import deque
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
except Exception:
    OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '')

if not OPENAI_API_KEY:
    raise ValueError("Set OPENAI_API_KEY in Colab secrets (key icon in sidebar).")

from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)

def chat(prompt: str, system: str = "You are a helpful assistant.", temperature: float = 0.7) -> str:
    """Simple wrapper: send a prompt, get text back."""
    resp = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}],
        temperature=temperature,
    )
    return resp.choices[0].message.content.strip()

def chat_messages(messages: list, temperature: float = 0.7, tools=None, tool_choice=None) -> Any:
    """Send a list of messages, optionally with tools. Returns the full response."""
    kwargs = dict(model="gpt-4o", messages=messages, temperature=temperature)
    if tools:
        kwargs["tools"] = tools
    if tool_choice:
        kwargs["tool_choice"] = tool_choice
    return client.chat.completions.create(**kwargs)

# Quick test
print(chat("Reply with exactly: ARCHITECTURES READY", temperature=0))

---
# 1. ReAct (Reason + Act)

The most widely used agent pattern. The LLM explicitly outputs its
**Thought** (reasoning), then an **Action** (tool call), then observes
the **Observation** (tool result), and repeats until it can answer.

```
User Question
  |
  v
THOUGHT: "We need to look up pricing info..."
ACTION:  search_kb("pricing")
OBSERVATION: "Pro plan: $99/mo..."
  |
  v
THOUGHT: "Now we can calculate annual cost..."
ACTION:  calculate("99 * 12")
OBSERVATION: "Result: 1188"
  |
  v
ANSWER: "The annual cost of Pro is $1,188."
```

**Why it is the default:** Simple, flexible, proven in production. Works for
80% of use cases. The explicit thought step makes the agent auditable.

**Weakness:** Can wander if the task is ambiguous. No upfront plan.

In [ ]:
# -- Architecture 1: ReAct Agent --

# Tools available to the agent
TOOLS = {
    "search_kb": lambda query: {
        "pricing": "Standard: $29/mo, Pro: $99/mo, Enterprise: custom.",
        "refund": "Full refund in 30 days. 50% in 60 days. Gold members: 90 days.",
        "features": "Core: dashboards, API. Pro: SSO, priority support.",
    }.get(query.lower().split()[0] if query else "", f"No results for '{query}'."),

    "calculate": lambda expr: str(eval(expr, {"__builtins__": {}})),

    "get_date": lambda _="": datetime.now().strftime("%Y-%m-%d"),
}

def react_agent(question: str, max_steps: int = 6) -> dict:
    tool_desc = "\n".join(f"- {n}: takes a string argument" for n in TOOLS)
    context = f"Question: {question}\n"
    trace = []

    for step in range(1, max_steps + 1):
        # THOUGHT
        thought = chat(
            f"{context}\nAvailable tools:\n{tool_desc}\n\n"
            f"Step {step} -- THOUGHT: Reason about what we know and what we need next. "
            f"If we can answer, start with FINAL ANSWER:",
            system="We are a ReAct agent. Think step by step. Be concise.",
            temperature=0.3,
        )
        trace.append({"step": step, "type": "thought", "content": thought})
        print(f"  [THOUGHT {step}] {thought[:200]}")

        if "FINAL ANSWER:" in thought.upper():
            idx = thought.upper().index("FINAL ANSWER:")
            answer = thought[idx + 13:].strip()
            return {"answer": answer, "trace": trace, "steps": step}

        # ACTION
        action_raw = chat(
            f"{context}\nThought: {thought}\n\n"
            f"Tools: {', '.join(TOOLS.keys())}\n"
            f"Choose a tool. Respond with JSON: {{\"tool\": \"name\", \"input\": \"value\"}}",
            temperature=0,
        )
        try:
            clean = action_raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
            action = json.loads(clean)
        except:
            action = {"tool": "search_kb", "input": question}

        tool_name = action.get("tool", "search_kb")
        tool_input = action.get("input", "")
        trace.append({"step": step, "type": "action", "tool": tool_name, "input": tool_input})
        print(f"  [ACTION  {step}] {tool_name}('{tool_input}')")

        # OBSERVATION
        result = TOOLS.get(tool_name, lambda x: "Unknown tool")(tool_input)
        trace.append({"step": step, "type": "observation", "content": str(result)})
        print(f"  [OBSERVE {step}] {str(result)[:150]}")

        context += f"\nThought: {thought}\nAction: {tool_name}('{tool_input}')\nObservation: {result}\n"

    return {"answer": "Max steps reached.", "trace": trace, "steps": max_steps}


# -- Demo --
print("=" * 60)
print("ReAct Agent Demo")
print("=" * 60)
result = react_agent("What is the annual cost of the Pro plan, and what features does it include?")
print(f"\nFinal Answer: {result['answer'][:300]}")
print(f"Steps taken: {result['steps']}")

---
# 2. Plan-Execute

Instead of interleaving thinking and acting (like ReAct), this pattern
**separates planning from execution**. First, the LLM creates a complete
plan. Then, a separate executor handles each step.

```
User Question
  |
  v
PLANNER:
  Step 1: Look up competitor pricing
  Step 2: Look up our pricing
  Step 3: Calculate difference
  Step 4: Write comparison report
  |
  v
EXECUTOR: runs each step (may use tools)
  |
  v
SYNTHESIZER: combines all results
```

**Where it shines:** Multi-step workflows where order matters --
research pipelines, report generation, automation scripts.

**Weakness:** If the plan is wrong, all downstream execution is wasted.
ReAct can course-correct mid-stream; Plan-Execute cannot (unless we add replanning).

In [ ]:
# -- Architecture 2: Plan-Execute Agent --

class PlanExecuteAgent:
    def __init__(self):
        self.plan = []
        self.results = []

    def _create_plan(self, task: str) -> list:
        raw = chat(
            f"Task: {task}\n\n"
            f"Create a step-by-step plan with 3-5 concrete steps. "
            f"Return ONLY a JSON list: [\"step 1\", \"step 2\", ...]",
            system="We are a strategic planner. Create clear, actionable plans.",
            temperature=0.3,
        )
        try:
            clean = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
            return json.loads(clean)
        except:
            return [raw]

    def _execute_step(self, step: str, context: str) -> str:
        return chat(
            f"Previous results:\n{context}\n\nCurrent step: {step}\n\n"
            f"Execute this step thoroughly. Provide concrete output.",
            system="We are a precise executor. Complete the step with specific details.",
            temperature=0.5,
        )

    def _synthesize(self, task: str) -> str:
        all_results = "\n\n".join(f"Step {i+1}: {r[:300]}" for i, r in enumerate(self.results))
        return chat(
            f"Task: {task}\n\nCompleted steps:\n{all_results}\n\n"
            f"Synthesize into a clear, comprehensive answer.",
            temperature=0.5,
        )

    def run(self, task: str) -> dict:
        # PLAN
        self.plan = self._create_plan(task)
        print(f"  [PLAN] {len(self.plan)} steps:")
        for i, s in enumerate(self.plan):
            print(f"    {i+1}. {s[:80]}")

        # EXECUTE
        self.results = []
        for i, step in enumerate(self.plan):
            context = "\n".join(f"[{j+1}] {r[:200]}" for j, r in enumerate(self.results))
            result = self._execute_step(step, context)
            self.results.append(result)
            print(f"  [EXEC {i+1}] {result[:120]}...")

        # SYNTHESIZE
        final = self._synthesize(task)
        return {"answer": final, "plan": self.plan, "step_results": self.results}


# -- Demo --
print("=" * 60)
print("Plan-Execute Agent Demo")
print("=" * 60)
agent = PlanExecuteAgent()
result = agent.run("Create a comparison of Python web frameworks (FastAPI, Django, Flask) for building a REST API.")
print(f"\nFinal Answer:\n{result['answer'][:500]}")

---
# 3. Reflection / Self-Critique Loop

The agent generates an output, then **evaluates its own work** and
improves it iteratively. This is "Do, Critique, Improve" in a loop.

```
Attempt 1 --> Evaluator (score: 5/10) --> Reflector (identify gaps)
                                              |
                                              v
Attempt 2 --> Evaluator (score: 7/10) --> Reflector (minor fixes)
                                              |
                                              v
Attempt 3 --> Evaluator (score: 9/10) --> ACCEPT
```

**Use case:** Tasks where quality matters more than speed -- code
generation, writing, complex reasoning, mathematical proofs.

**Tradeoff:** 2-3x more LLM calls = higher cost and latency. Worth it
when accuracy is critical.

In [ ]:
# -- Architecture 3: Reflection / Self-Critique Agent --

class ReflectionAgent:
    def __init__(self):
        self.attempts = []

    def _generate(self, task: str, feedback: str = "") -> str:
        prompt = f"Task: {task}"
        if feedback:
            prompt += f"\n\nPrevious feedback to address:\n{feedback}"
        return chat(prompt, system="We are an expert analyst. Be thorough and precise.", temperature=0.7)

    def _evaluate(self, task: str, output: str) -> dict:
        raw = chat(
            f"Task: {task}\n\nOutput to evaluate:\n{output[:600]}\n\n"
            f"Score this output 1-10 on: completeness, accuracy, clarity, actionability. "
            f"Return JSON: {{\"score\": N, \"strengths\": \"...\", \"weaknesses\": \"...\", \"suggestions\": \"...\"}}",
            system="We are a strict but fair evaluator.", temperature=0.3,
        )
        try:
            clean = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
            return json.loads(clean)
        except:
            return {"score": 5, "strengths": "", "weaknesses": raw, "suggestions": ""}

    def _reflect(self, task: str, output: str, evaluation: dict) -> str:
        return chat(
            f"Task: {task}\n\nOutput:\n{output[:400]}\n\n"
            f"Evaluation: Score {evaluation['score']}/10\n"
            f"Weaknesses: {evaluation.get('weaknesses', '')}\n"
            f"Suggestions: {evaluation.get('suggestions', '')}\n\n"
            f"What specific changes should be made? List 3 concrete improvements.",
            system="We are a reflective improver. Be specific.", temperature=0.3,
        )

    def run(self, task: str, max_attempts: int = 3, threshold: int = 8) -> dict:
        feedback = ""
        for attempt in range(1, max_attempts + 1):
            # GENERATE
            output = self._generate(task, feedback)
            print(f"  [ATTEMPT {attempt}] {output[:120]}...")

            # EVALUATE
            evaluation = self._evaluate(task, output)
            score = evaluation.get("score", 0)
            print(f"  [SCORE  {attempt}] {score}/10 -- {evaluation.get('weaknesses', '')[:100]}")

            self.attempts.append({"output": output, "score": score, "evaluation": evaluation})

            if score >= threshold:
                print(f"  [ACCEPT] Score {score}/10 meets threshold.")
                return {"answer": output, "attempts": len(self.attempts), "final_score": score}

            # REFLECT
            feedback = self._reflect(task, output, evaluation)
            print(f"  [REFLECT {attempt}] {feedback[:120]}...")

        best = max(self.attempts, key=lambda a: a["score"])
        return {"answer": best["output"], "attempts": len(self.attempts), "final_score": best["score"]}


# -- Demo --
print("=" * 60)
print("Reflection Agent Demo")
print("=" * 60)
agent = ReflectionAgent()
result = agent.run("Write a technical explanation of how database indexing works, including B-tree indexes and when to use them.")
print(f"\nFinal Answer ({result['attempts']} attempts, score {result['final_score']}/10):")
print(result['answer'][:500])

---
# 4. Multi-Agent Systems

Multiple specialized agents, each with its own system prompt and expertise,
collaborating on a shared task. Common patterns:

- **Parallel:** agents work independently, results are merged
- **Sequential:** output of one feeds into the next
- **Supervisor:** a lead agent delegates and synthesizes

```
User Question
  |
  v
SUPERVISOR
  |--- Researcher --> research findings
  |--- Analyst    --> data analysis
  |--- Writer     --> draft report
  |
  v
SUPERVISOR synthesizes --> Final Report
```

**Why it matters:** Scales to complex problems by decomposition.

**Reality check:** Coordination overhead is real. Agents can contradict
each other, and the supervisor must handle conflicts. Start simple.

In [ ]:
# -- Architecture 4: Multi-Agent System --

class SpecialistAgent:
    """An agent with a specific role and isolated context."""
    def __init__(self, name: str, role: str):
        self.name = name
        self.role = role
        self.history = []

    def respond(self, task: str) -> str:
        self.history.append({"role": "user", "content": task})
        resp = chat(task, system=self.role, temperature=0.7)
        self.history.append({"role": "assistant", "content": resp})
        return resp

class SupervisorSystem:
    def __init__(self, specialists: list):
        self.specialists = {s.name: s for s in specialists}
        self.supervisor = SpecialistAgent(
            "Supervisor",
            "We are a lead analyst. We synthesize findings from multiple specialists "
            "into a coherent, well-structured executive summary. We resolve contradictions "
            "and highlight consensus."
        )

    def run(self, task: str, parallel: bool = True) -> dict:
        results = {}

        if parallel:
            def _run(agent):
                return agent.name, agent.respond(f"As the {agent.name}, analyze: {task}")
            with ThreadPoolExecutor(max_workers=len(self.specialists)) as pool:
                results = dict(pool.map(_run, self.specialists.values()))
        else:
            for name, agent in self.specialists.items():
                results[name] = agent.respond(f"As the {name}, analyze: {task}")

        for name, output in results.items():
            print(f"  [{name.upper()}] {output[:150]}...")

        # Supervisor synthesizes
        combined = "\n\n".join(f"[{n}]\n{o[:400]}" for n, o in results.items())
        synthesis = self.supervisor.respond(
            f"Task: {task}\n\nSpecialist findings:\n{combined}\n\n"
            f"Synthesize into a unified analysis. Note any disagreements."
        )
        results["SYNTHESIS"] = synthesis
        return results


# -- Demo --
print("=" * 60)
print("Multi-Agent System Demo")
print("=" * 60)

system = SupervisorSystem([
    SpecialistAgent("Market Researcher", "We are a market researcher. We focus on market size, trends, and growth projections. 4 sentences max."),
    SpecialistAgent("Technical Architect", "We are a technical architect. We focus on system design, scalability, and tech stack choices. 4 sentences max."),
    SpecialistAgent("Risk Analyst", "We are a risk analyst. We focus on potential failures, security risks, and mitigation strategies. 4 sentences max."),
])

result = system.run("Evaluate the feasibility of building a real-time fraud detection system using LLMs.")
print(f"\n[SYNTHESIS]\n{result['SYNTHESIS'][:500]}")

---
# 5. Tool-Augmented Agents (Function Calling)

This is what **production systems actually use**. Instead of parsing
free-text tool calls (like ReAct), we use the LLM's native **function
calling** capability. The LLM outputs a structured JSON tool call,
we execute it, and feed the result back.

```
User: "What is the weather in Tokyo?"
  |
  v
LLM response: tool_call(get_weather, {"city": "Tokyo"})
  |
  v
Execute: get_weather("Tokyo") --> "22C, partly cloudy"
  |
  v
LLM response: "The weather in Tokyo is 22C and partly cloudy."
```

**Why this is the production standard:** The tool call format is
structured and validated -- no regex parsing of free text. OpenAI,
Anthropic, and Google all support this natively.

**Key difference from ReAct:** ReAct is a prompting pattern (any LLM).
Function calling is a model-level feature (specific APIs).

In [ ]:
# -- Architecture 5: Tool-Augmented Agent (Native Function Calling) --

# Define tools using OpenAI's function calling schema
TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "search_knowledge_base",
            "description": "Search our internal knowledge base for company information.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search query"}
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression, e.g. '99 * 12'"}
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "send_email",
            "description": "Send an email to a recipient.",
            "parameters": {
                "type": "object",
                "properties": {
                    "to": {"type": "string", "description": "Recipient email"},
                    "subject": {"type": "string", "description": "Email subject"},
                    "body": {"type": "string", "description": "Email body"},
                },
                "required": ["to", "subject", "body"],
            },
        },
    },
]

# Tool implementations
TOOL_FUNCS = {
    "search_knowledge_base": lambda query: {
        "pricing": "Pro: $99/mo, Standard: $29/mo, Enterprise: custom.",
        "refund": "Full refund in 30 days. Gold: 90 days.",
        "features": "Core: API, dashboards. Pro: SSO, priority support.",
    }.get(query.lower().split()[0] if query else "", f"No results for '{query}'."),
    "calculate": lambda expression: str(eval(expression, {"__builtins__": {}})),
    "send_email": lambda to, subject, body: f"Email sent to {to}: '{subject}'",
}

def function_calling_agent(question: str, max_turns: int = 5) -> dict:
    messages = [
        {"role": "system", "content": "We are a helpful assistant. Use tools when needed."},
        {"role": "user", "content": question},
    ]
    tool_log = []

    for turn in range(max_turns):
        resp = chat_messages(messages, tools=TOOL_SCHEMAS, temperature=0.3)
        msg = resp.choices[0].message

        if not msg.tool_calls:
            return {"answer": msg.content, "tool_calls": tool_log}

        messages.append(msg)
        for tc in msg.tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments)
            tool_log.append({"tool": fn_name, "args": fn_args})
            print(f"  [TOOL CALL] {fn_name}({fn_args})")

            result = TOOL_FUNCS[fn_name](**fn_args)
            print(f"  [RESULT]    {str(result)[:120]}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})

    return {"answer": "Max turns.", "tool_calls": tool_log}


# -- Demo --
print("=" * 60)
print("Tool-Augmented Agent (Function Calling) Demo")
print("=" * 60)

for q in [
    "What is the annual cost of our Pro plan?",
    "Look up our refund policy, then email support@acme.com a summary.",
]:
    print(f"\nQ: {q}")
    result = function_calling_agent(q)
    print(f"Answer: {result['answer'][:250]}")
    print(f"Tools used: {len(result['tool_calls'])}")

---
# 6. Memory-Augmented Architecture

Without memory, the agent forgets everything between calls. Memory-augmented
agents maintain state across interactions:

| Memory Type | What It Stores | Analogy |
|-------------|---------------|---------|
| **Buffer (short-term)** | Last N messages | Working attention |
| **Semantic (long-term)** | Extracted facts | Knowledge base |
| **Episodic** | Past interactions with timestamps | Personal experiences |

```
User: "My name is Alex, I work at Acme Corp."
  |
  v
SEMANTIC MEMORY stores: {user: ["name is Alex", "works at Acme Corp"]}
  |
  v
[Later] User: "What do you know about me?"
  |
  v
Agent recalls from semantic memory --> "You are Alex from Acme Corp."
```

**Without this, the agent is goldfish-tier.** Production agents need
at least buffer + semantic memory.

In [ ]:
# -- Architecture 6: Memory-Augmented Agent --

class MemoryAgent:
    def __init__(self):
        # Buffer: last N messages
        self.buffer = deque(maxlen=8)
        # Semantic: extracted facts (entity -> facts)
        self.semantic: Dict[str, list] = {}
        # Episodic: timestamped interaction logs
        self.episodic: list = []

    def _extract_facts(self, text: str):
        """Use LLM to extract structured facts from user input."""
        if not any(kw in text.lower() for kw in ["my name", "i work", "i prefer", "we use", "our"]):
            return
        raw = chat(
            f"Extract facts from: '{text}'. Return JSON: {{\"entity\": [\"fact1\"]}}",
            temperature=0,
        )
        try:
            clean = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
            for entity, facts in json.loads(clean).items():
                self.semantic.setdefault(entity, [])
                for f in facts:
                    if f not in self.semantic[entity]:
                        self.semantic[entity].append(f)
        except:
            pass

    def _build_context(self) -> str:
        parts = ["We are a helpful assistant with memory."]
        # Semantic facts
        if self.semantic:
            facts = "\n".join(f"  {e}: {', '.join(fs)}" for e, fs in self.semantic.items())
            parts.append(f"Known facts:\n{facts}")
        # Recent episodic
        if self.episodic:
            recent = self.episodic[-3:]
            eps = "\n".join(f"  [{e['ts']}] Q: {e['q'][:60]}" for e in recent)
            parts.append(f"Recent interactions:\n{eps}")
        return "\n\n".join(parts)

    def chat(self, user_input: str) -> str:
        self.buffer.append({"role": "user", "content": user_input})
        self._extract_facts(user_input)

        messages = [{"role": "system", "content": self._build_context()}]
        messages += list(self.buffer)

        resp = chat_messages(messages, temperature=0.7)
        answer = resp.choices[0].message.content.strip()

        self.buffer.append({"role": "assistant", "content": answer})
        self.episodic.append({"ts": datetime.now().strftime("%H:%M:%S"), "q": user_input, "a": answer})
        return answer


# -- Demo --
print("=" * 60)
print("Memory-Augmented Agent Demo")
print("=" * 60)

agent = MemoryAgent()
conversation = [
    "My name is Alex and I work at Acme Corp. We use Python and PostgreSQL.",
    "What web framework would you recommend for our stack?",
    "Wait -- what do you know about me and our stack?",
]

for msg in conversation:
    print(f"\n[USER] {msg}")
    resp = agent.chat(msg)
    print(f"[AGENT] {resp[:250]}")

print(f"\n--- Memory State ---")
print(f"Buffer: {len(agent.buffer)} messages")
print(f"Semantic: {json.dumps(agent.semantic, indent=2)}")
print(f"Episodic: {len(agent.episodic)} interactions")

---
# 7. Event-Driven Agents

Instead of waiting for user input, these agents are **triggered by
external events** -- webhooks, message queues, cron jobs, or system
notifications. Highly practical for real-world automation.

```
EVENT: new_email(from="client@co.com", subject="Invoice Q3")
  |
  v
ROUTER: classifies event type --> "finance"
  |
  v
FINANCE HANDLER: extracts invoice data, updates CRM, sends confirmation
```

**Examples:**
- New email --> agent drafts a reply
- New lead in CRM --> agent qualifies and routes
- Error alert --> agent diagnoses and creates ticket
- Scheduled --> agent generates daily report

**Why this matters:** This is how agents integrate into existing
business workflows. The agent is not a chatbot -- it is a background worker.

In [ ]:
# -- Architecture 7: Event-Driven Agent --

class EventDrivenAgent:
    def __init__(self):
        self.handlers: Dict[str, Callable] = {}
        self.event_log: list = []

    def register(self, event_type: str, handler: Callable):
        self.handlers[event_type] = handler

    def _classify_event(self, event: dict) -> str:
        raw = chat(
            f"Classify this event into one category: {list(self.handlers.keys())}\n\n"
            f"Event: {json.dumps(event)}\n\nReturn ONLY the category name.",
            temperature=0,
        )
        category = raw.strip().strip('"').lower()
        # Match to registered handler
        for key in self.handlers:
            if key in category:
                return key
        return list(self.handlers.keys())[0]  # fallback

    def process(self, event: dict) -> dict:
        event_type = self._classify_event(event)
        handler = self.handlers.get(event_type)
        print(f"  [EVENT]    {event.get('type', 'unknown')} --> classified as '{event_type}'")

        if handler:
            result = handler(event)
            self.event_log.append({"event": event, "type": event_type, "result": result})
            print(f"  [HANDLER]  {result[:150]}")
            return {"type": event_type, "result": result}
        return {"type": event_type, "result": "No handler found."}


# Handlers
def handle_email(event: dict) -> str:
    return chat(
        f"Draft a brief professional reply to this email:\n"
        f"From: {event.get('from', 'unknown')}\nSubject: {event.get('subject', '')}\n"
        f"Body: {event.get('body', '')}",
        system="We draft professional, concise email replies.", temperature=0.5,
    )

def handle_alert(event: dict) -> str:
    return chat(
        f"Diagnose this alert and suggest a fix:\n"
        f"Service: {event.get('service', 'unknown')}\n"
        f"Error: {event.get('error', '')}\nSeverity: {event.get('severity', 'medium')}",
        system="We are a DevOps engineer. Diagnose concisely.", temperature=0.3,
    )

def handle_lead(event: dict) -> str:
    return chat(
        f"Qualify this sales lead and suggest next action:\n"
        f"Company: {event.get('company', '')}\nSize: {event.get('size', '')}\n"
        f"Interest: {event.get('interest', '')}",
        system="We are a sales qualification expert. Be direct.", temperature=0.5,
    )


# -- Demo --
print("=" * 60)
print("Event-Driven Agent Demo")
print("=" * 60)

agent = EventDrivenAgent()
agent.register("email", handle_email)
agent.register("alert", handle_alert)
agent.register("lead", handle_lead)

events = [
    {"type": "incoming_email", "from": "client@bigcorp.com", "subject": "API Integration Timeline",
     "body": "Hi, when can we expect the API to be ready for integration testing?"},
    {"type": "system_alert", "service": "payment-gateway", "error": "Connection timeout to Stripe API",
     "severity": "high"},
    {"type": "new_crm_lead", "company": "TechStartup Inc", "size": "50 employees",
     "interest": "Enterprise plan for AI analytics"},
]

for event in events:
    print(f"\n--- Processing: {event.get('type', 'unknown')} ---")
    result = agent.process(event)

print(f"\nTotal events processed: {len(agent.event_log)}")

---
# 8. Graph-Based Execution (LangGraph Style)

Model the agent as a **directed graph** where nodes are processing steps
and edges are transitions (including conditional branches). This gives us
deterministic, controllable execution -- the agent follows a defined path
rather than improvising.

```
START --> classify --> [technical] --> deep_analysis --> review --> END
                   --> [simple]    --> quick_answer  -----------> END
```

**Why it is rising:** Production reliability. Unlike ReAct (which can
wander), graph-based execution guarantees the agent follows a known path.
We can add retries, timeouts, and fallbacks at specific nodes.

**Key insight:** This is not mutually exclusive with other patterns.
We can put a ReAct loop inside a graph node, or use reflection at a
specific step. The graph is the orchestration layer.

In [ ]:
# -- Architecture 8: Graph-Based Execution --

class GraphNode:
    def __init__(self, name: str, fn: Callable):
        self.name = name
        self.fn = fn

class AgentGraph:
    """A simple directed graph executor with conditional edges."""
    def __init__(self):
        self.nodes: Dict[str, GraphNode] = {}
        self.edges: Dict[str, Any] = {}  # node_name -> next_name or callable
        self.entry: str = ""

    def add_node(self, name: str, fn: Callable):
        self.nodes[name] = GraphNode(name, fn)

    def add_edge(self, from_node: str, to_node: str):
        self.edges[from_node] = to_node

    def add_conditional_edge(self, from_node: str, router: Callable):
        self.edges[from_node] = router

    def set_entry(self, name: str):
        self.entry = name

    def run(self, state: dict) -> dict:
        current = self.entry
        visited = []

        while current and current != "END":
            node = self.nodes.get(current)
            if not node:
                break
            print(f"  [NODE] {node.name}")
            state = {**state, **node.fn(state)}
            visited.append(current)

            # Resolve next node
            edge = self.edges.get(current, "END")
            if callable(edge):
                current = edge(state)
                print(f"  [ROUTE] --> {current}")
            else:
                current = edge

        state["_visited"] = visited
        return state

# Build a graph for technical support
def classify_node(state: dict) -> dict:
    category = chat(
        f"Classify this query as 'technical', 'billing', or 'general'. Reply with ONE word only.\n"
        f"Query: {state['query']}",
        temperature=0,
    )
    cat = "technical" if "tech" in category.lower() else "billing" if "bill" in category.lower() else "general"
    return {"category": cat}

def technical_node(state: dict) -> dict:
    analysis = chat(
        f"Provide a detailed technical analysis for: {state['query']}",
        system="We are a senior technical support engineer.", temperature=0.5,
    )
    return {"analysis": analysis}

def billing_node(state: dict) -> dict:
    analysis = chat(
        f"Address this billing question: {state['query']}\n"
        f"Our pricing: Standard $29/mo, Pro $99/mo, Enterprise custom.",
        system="We are a billing specialist.", temperature=0.5,
    )
    return {"analysis": analysis}

def general_node(state: dict) -> dict:
    analysis = chat(f"Provide a helpful answer to: {state['query']}", temperature=0.5)
    return {"analysis": analysis}

def review_node(state: dict) -> dict:
    review = chat(
        f"Review this response for accuracy and completeness. Improve if needed.\n"
        f"Query: {state['query']}\nDraft: {state['analysis'][:400]}",
        system="We are a QA reviewer.", temperature=0.3,
    )
    return {"final_response": review}

def route_by_category(state: dict) -> str:
    return {"technical": "technical", "billing": "billing"}.get(state["category"], "general")

# Assemble graph
graph = AgentGraph()
graph.add_node("classify", classify_node)
graph.add_node("technical", technical_node)
graph.add_node("billing", billing_node)
graph.add_node("general", general_node)
graph.add_node("review", review_node)

graph.set_entry("classify")
graph.add_conditional_edge("classify", route_by_category)
graph.add_edge("technical", "review")
graph.add_edge("billing", "review")
graph.add_edge("general", "review")
graph.add_edge("review", "END")

# -- Demo --
print("=" * 60)
print("Graph-Based Execution Demo")
print("=" * 60)

for q in [
    "Why is my API returning 429 status codes?",
    "What is the annual cost of the Pro plan?",
]:
    print(f"\nQ: {q}")
    result = graph.run({"query": q})
    print(f"  Route: {' -> '.join(result['_visited'])}")
    print(f"  Answer: {result.get('final_response', '')[:250]}")

---
# 9. Autonomous Loop Agents

The agent runs in a loop **until it decides its goal is achieved**,
with no fixed number of steps. It continuously reasons, acts, and
evaluates progress.

```
GOAL: "Research and write a report on quantum computing market."
  |
  v
while not goal_achieved:
    plan = think_about_next_action()
    result = execute(plan)
    progress = evaluate_progress()
    if progress >= threshold: break
```

**Sounds cool. Mostly dangerous.** Without guardrails:
- Infinite loops
- Cost explosions (hundreds of LLM calls)
- Hallucination spirals

**Use carefully** with: max iterations, cost limits, progress tracking,
and human-in-the-loop checkpoints.

In [ ]:
# -- Architecture 9: Autonomous Loop Agent --

class AutonomousAgent:
    """Goal-directed agent with safety guardrails."""
    def __init__(self, max_iterations: int = 6, cost_limit: int = 10):
        self.max_iterations = max_iterations
        self.cost_limit = cost_limit
        self.iteration_log = []
        self.calls_made = 0

    def _think(self, goal: str, history: str) -> str:
        self.calls_made += 1
        return chat(
            f"Goal: {goal}\n\nProgress so far:\n{history}\n\n"
            f"What should we do next to make progress toward the goal? "
            f"If the goal is achieved, say GOAL ACHIEVED.",
            system="We are a goal-oriented agent. Be strategic and efficient.",
            temperature=0.5,
        )

    def _act(self, action_plan: str, goal: str) -> str:
        self.calls_made += 1
        return chat(
            f"Goal: {goal}\nAction: {action_plan}\n\n"
            f"Execute this action. Provide concrete output.",
            temperature=0.7,
        )

    def _evaluate(self, goal: str, history: str) -> dict:
        self.calls_made += 1
        raw = chat(
            f"Goal: {goal}\n\nWork done:\n{history}\n\n"
            f"Rate progress 1-10. Return JSON: "
            f"{{\"progress\": N, \"remaining\": \"what is left to do\"}}",
            temperature=0.3,
        )
        try:
            clean = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
            return json.loads(clean)
        except:
            return {"progress": 5, "remaining": raw}

    def run(self, goal: str) -> dict:
        history = ""
        for i in range(1, self.max_iterations + 1):
            # Safety check
            if self.calls_made >= self.cost_limit:
                print(f"  [GUARDRAIL] Cost limit ({self.cost_limit} calls) reached.")
                break

            # THINK
            thought = self._think(goal, history or "Nothing yet.")
            print(f"  [ITER {i} THINK] {thought[:120]}...")

            if "GOAL ACHIEVED" in thought.upper():
                print(f"  [DONE] Agent reports goal achieved at iteration {i}.")
                return {"result": history, "iterations": i, "calls": self.calls_made, "status": "achieved"}

            # ACT
            result = self._act(thought, goal)
            history += f"\nIteration {i}: {result[:300]}"
            print(f"  [ITER {i} ACT]   {result[:120]}...")

            # EVALUATE
            progress = self._evaluate(goal, history)
            score = progress.get("progress", 0)
            remaining = progress.get("remaining", "")
            print(f"  [ITER {i} EVAL]  Progress: {score}/10 -- {remaining[:80]}")

            self.iteration_log.append({"iteration": i, "score": score, "remaining": remaining})

            if score >= 9:
                print(f"  [DONE] Progress threshold met at iteration {i}.")
                return {"result": history, "iterations": i, "calls": self.calls_made, "status": "achieved"}

        return {"result": history, "iterations": self.max_iterations, "calls": self.calls_made, "status": "max_iter"}


# -- Demo --
print("=" * 60)
print("Autonomous Loop Agent Demo (with guardrails)")
print("=" * 60)

agent = AutonomousAgent(max_iterations=4, cost_limit=15)
result = agent.run("Write a concise analysis of three key trends in AI for 2025: agents, multimodal models, and reasoning.")
print(f"\nStatus: {result['status']} | Iterations: {result['iterations']} | LLM calls: {result['calls']}")
print(f"Result preview: {result['result'][:400]}")

---
# 10. Hierarchical Agents

A **manager agent** decomposes a complex task and delegates sub-tasks
to specialized sub-agents. Each sub-agent may itself delegate further,
creating a tree of responsibility.

```
MANAGER (CEO-level view)
  |
  |--- SubAgent: Data Collection
  |       |--- worker: scrape sources
  |       |--- worker: clean data
  |
  |--- SubAgent: Analysis
  |       |--- worker: statistical analysis
  |       |--- worker: visualization
  |
  |--- SubAgent: Report Writing
```

**Good for:** Large-scale enterprise workflows, complex pipelines
with clear decomposition.

**Overhead:** More coordination logic, more LLM calls, harder to debug.
Each layer adds latency. Use only when the task genuinely requires
this level of structure.

In [ ]:
# -- Architecture 10: Hierarchical Agents --

class WorkerAgent:
    def __init__(self, name: str, skill: str):
        self.name = name
        self.skill = skill

    def execute(self, task: str) -> str:
        return chat(
            f"Task: {task}",
            system=f"We are a {self.skill} specialist. Be thorough but concise.",
            temperature=0.7,
        )

class TeamLead:
    def __init__(self, name: str, domain: str, workers: list):
        self.name = name
        self.domain = domain
        self.workers = {w.name: w for w in workers}

    def handle(self, task: str) -> dict:
        # Decompose task for workers
        raw = chat(
            f"Task: {task}\n\nAvailable workers: {list(self.workers.keys())}\n"
            f"Assign each worker a specific sub-task. Return JSON: "
            f"{{\"worker_name\": \"sub-task description\"}}",
            system=f"We are the {self.domain} team lead. Delegate efficiently.",
            temperature=0.3,
        )
        try:
            clean = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
            assignments = json.loads(clean)
        except:
            assignments = {list(self.workers.keys())[0]: task}

        results = {}
        for worker_name, sub_task in assignments.items():
            worker = self.workers.get(worker_name)
            if worker:
                results[worker_name] = worker.execute(sub_task)
                print(f"    [{worker_name}] {results[worker_name][:100]}...")
        return results

class ManagerAgent:
    def __init__(self, teams: list):
        self.teams = {t.name: t for t in teams}

    def run(self, task: str) -> dict:
        # Step 1: Manager decomposes
        raw = chat(
            f"Task: {task}\n\nAvailable teams: {list(self.teams.keys())}\n"
            f"Decompose into team-level tasks. Return JSON: "
            f"{{\"team_name\": \"team-level task\"}}",
            system="We are a senior manager. Decompose strategically.",
            temperature=0.3,
        )
        try:
            clean = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
            team_tasks = json.loads(clean)
        except:
            team_tasks = {list(self.teams.keys())[0]: task}

        # Step 2: Each team handles its task
        all_results = {}
        for team_name, team_task in team_tasks.items():
            team = self.teams.get(team_name)
            if team:
                print(f"  [TEAM: {team_name}] {team_task[:80]}")
                all_results[team_name] = team.handle(team_task)

        # Step 3: Manager synthesizes
        flat_results = "\n".join(
            f"[{team}] {worker}: {output[:200]}"
            for team, workers in all_results.items()
            for worker, output in workers.items()
        )
        synthesis = chat(
            f"Task: {task}\n\nTeam outputs:\n{flat_results}\n\n"
            f"Synthesize into a final deliverable.",
            system="We are the senior manager. Create a polished final output.",
            temperature=0.5,
        )
        return {"synthesis": synthesis, "team_results": all_results}

# -- Demo --
print("=" * 60)
print("Hierarchical Agent Demo")
print("=" * 60)

manager = ManagerAgent([
    TeamLead("Research", "research", [
        WorkerAgent("data_collector", "data collection and sourcing"),
        WorkerAgent("fact_checker", "fact verification and accuracy"),
    ]),
    TeamLead("Analysis", "analysis", [
        WorkerAgent("trend_analyst", "trend analysis and pattern recognition"),
        WorkerAgent("risk_assessor", "risk assessment and mitigation"),
    ]),
    TeamLead("Writing", "content", [
        WorkerAgent("writer", "technical writing and report creation"),
        WorkerAgent("editor", "editing and quality assurance"),
    ]),
])

result = manager.run("Create a brief report on the current state of AI-powered code generation tools.")
print(f"\n[FINAL SYNTHESIS]\n{result['synthesis'][:500]}")

---
# Summary: When to Use What

| Architecture | Use When | Avoid When |
|-------------|----------|------------|
| **ReAct** | General purpose, tool-using tasks | Need deterministic flow |
| **Plan-Execute** | Multi-step, ordered workflows | Plan might be wrong |
| **Reflection** | Quality is critical (code, reasoning) | Speed matters more |
| **Multi-Agent** | Complex domain, clear specializations | Simple tasks (overkill) |
| **Tool-Augmented** | Production systems with APIs | No tools needed |
| **Memory-Augmented** | Stateful, personalized interactions | Stateless one-shot tasks |
| **Event-Driven** | Background automation, integrations | Interactive chat |
| **Graph-Based** | Need reliability and control | Exploratory, open-ended |
| **Autonomous Loop** | Open-ended goals with guardrails | Budget-constrained |
| **Hierarchical** | Enterprise scale, large teams | Small tasks (overhead) |

---

## Production Reality Check

Most production agent systems combine 2-3 of these patterns:

- **Common stack:** Graph-Based execution + Tool-Augmented + Memory
- **High-quality stack:** Graph-Based + Reflection at key nodes + Memory
- **Enterprise stack:** Hierarchical + Multi-Agent + Event-Driven

The patterns are not mutually exclusive -- they are composable building blocks.

---

## What Actually Works in Production

1. **Start with Tool-Augmented (Function Calling)** -- this alone covers most use cases
2. **Add Graph-Based execution** when we need deterministic flows
3. **Add Memory** when interactions need to be stateful
4. **Add Reflection** at quality-critical nodes, not everywhere
5. **Multi-Agent and Hierarchical** only when complexity genuinely demands it

Everything else is for experimentation until we have the guardrails to make it safe.